# Toy Test: Manual Early Forward Instead of Pop/Restore

This notebook replaces the old draft function:

```python
pop layers -> model forward -> restore layers
```

with:

```python
embed tokens -> manually run first N transformer blocks -> TunedLens -> sample draft token
```

Goal: test whether removing layer mutation and avoiding `output_hidden_states=True` improves runtime on T4.

## 1. Imports and setup

Do **not** hardcode a Hugging Face token in shared notebooks. Use an environment variable instead.

In [ ]:
import os
import time
import inspect
import numpy as np
import torch
import transformers

from datasets import load_dataset
from tuned_lens.nn.lenses import TunedLens
from transformers import AutoModelForCausalLM, AutoTokenizer

device_type = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "EleutherAI/pythia-1.4b-deduped"

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")

print("device:", device_type)
print("HF token exists:", bool(hf_token))
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Load model, tokenizer, TunedLens, and dataset

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if device_type.type == "cuda" else torch.float32,
    token=hf_token,
).to(device_type)

model.eval()

tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)

tuned_lens = TunedLens.from_model_and_pretrained(model).to(device_type)
tuned_lens.eval()

ds = load_dataset(
    "HuggingFaceFW/fineweb",
    "CC-MAIN-2014-10",
    split="train",
    streaming=True,
)

print("num layers:", len(model.gpt_neox.layers))
print("expected:", model.config.num_hidden_layers)
print("vocab size:", model.config.vocab_size)

## 3. Timing helper

CUDA is asynchronous, so we need `torch.cuda.synchronize()` before and after timed regions.

In [ ]:
def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def now():
    sync_cuda()
    return time.time()

## 4. Manual early forward for GPT-NeoX / Pythia

This function does **not** mutate the model.

It only runs:

```text
embed_in
→ transformer layers 0..layer_index
→ early hidden state
```

Since our prompts have no padding inside the sequence, we pass `attention_mask=None` inside the layer loop. GPT-NeoX attention still applies its internal causal mask.

In [ ]:
def gpt_neox_manual_early_hidden(model, input_ids, layer_index):
    """
    Run only GPT-NeoX/Pythia layers 0..layer_index.
    No pop/restore. No output_hidden_states=True.
    """
    gpt = model.gpt_neox
    device = input_ids.device
    B, T = input_ids.shape

    hidden_states = gpt.embed_in(input_ids)  # [B, T, hidden_dim]

    position_ids = torch.arange(T, device=device).unsqueeze(0)  # [1, T]

    for layer in gpt.layers[: layer_index + 1]:
        out = layer(
            hidden_states,
            attention_mask=None,
            position_ids=position_ids,
            head_mask=None,
            use_cache=False,
            output_attentions=False,
        )
        hidden_states = out[0]

    return hidden_states

## 5. New draft function: no pop, no restore, no all hidden states

Changes versus old function:

1. No `layers.pop()`
2. No `layers.extend()`
3. No `output_hidden_states=True`
4. TunedLens is applied only to the last token hidden state: `[B, 1, hidden_dim]`

In [ ]:
def generate_with_tuned_lens_manual(model, tuned_lens, layer_index, prefix_tokens, gamma):
    """
    Draft gamma tokens using manual early forward + TunedLens.

    Returns:
        {
          "sequences": [1, T + gamma],
          "scores": list of gamma tensors, each [1, vocab]
        }
    """
    scores = []

    for _ in range(gamma):
        input_ids = prefix_tokens[None]  # [1, T]

        h = gpt_neox_manual_early_hidden(
            model=model,
            input_ids=input_ids,
            layer_index=layer_index,
        )  # [1, T, hidden_dim]

        # Only the last position is needed for next-token prediction.
        last_h = h[:, -1:, :]  # [1, 1, hidden_dim]

        tuned_lens_logits = tuned_lens(last_h, layer_index)  # [1, 1, vocab]
        next_token_logits = tuned_lens_logits[:, -1, :]      # [1, vocab]

        p_for_next_token = torch.softmax(next_token_logits.float(), dim=-1)[0]
        next_token = torch.multinomial(p_for_next_token, num_samples=1)

        scores.append(next_token_logits)
        prefix_tokens = torch.cat([prefix_tokens, next_token])

    return {
        "sequences": prefix_tokens[None],
        "scores": scores,
    }

## 6. Quick correctness/shape smoke test

In [ ]:
example = next(iter(ds))
input_text = example["text"]

num_prefix_tokens = 10
layer_index = 2
gamma = 4

input_ids = tokenizer(input_text, return_tensors="pt")["input_ids"].to(device_type)[0][:num_prefix_tokens]

with torch.no_grad():
    out = generate_with_tuned_lens_manual(
        model=model,
        tuned_lens=tuned_lens,
        layer_index=layer_index,
        prefix_tokens=input_ids,
        gamma=gamma,
    )

print("input length:", len(input_ids))
print("output shape:", tuple(out["sequences"].shape))
print("num scores:", len(out["scores"]))
print("score shape:", tuple(out["scores"][0].shape))
print("draft text:")
print(tokenizer.decode(out["sequences"][0]))
print("model still full:", len(model.gpt_neox.layers), "/", model.config.num_hidden_layers)

## 7. Optional: compare old pop/restore draft time vs new manual draft time

Only run this if your model is currently full. This shows the overhead difference for the draft function alone.

In [ ]:
def generate_with_tuned_lens_pop_restore(model, tuned_lens, layer_index, prefix_tokens, gamma):
    layers = model.gpt_neox.layers

    if len(layers) != model.config.num_hidden_layers:
        raise ValueError(
            f"Model is already truncated: len(layers)={len(layers)}, "
            f"expected={model.config.num_hidden_layers}. Reload the model."
        )

    num_layers_to_keep = layer_index + 1
    num_layers_to_remove = len(layers) - num_layers_to_keep

    removed_layers = [layers.pop(-1) for _ in range(num_layers_to_remove)]
    scores = []

    for _ in range(gamma):
        out = model(
            prefix_tokens[None],
            output_hidden_states=True,
            return_dict=True,
        )

        h = out.hidden_states[layer_index + 1]
        last_h = h[:, -1:, :]

        tuned_lens_logits = tuned_lens(last_h, layer_index)
        next_token_logits = tuned_lens_logits[:, -1, :]

        p_for_next_token = torch.softmax(next_token_logits.float(), dim=-1)[0]
        next_token = torch.multinomial(p_for_next_token, num_samples=1)

        scores.append(next_token_logits)
        prefix_tokens = torch.cat([prefix_tokens, next_token])

    layers.extend(reversed(removed_layers))

    return {
        "sequences": prefix_tokens[None],
        "scores": scores,
    }


with torch.no_grad():
    # warmup
    _ = generate_with_tuned_lens_manual(model, tuned_lens, layer_index, input_ids, gamma)

    t0 = now()
    _ = generate_with_tuned_lens_manual(model, tuned_lens, layer_index, input_ids, gamma)
    manual_time = now() - t0

    t0 = now()
    _ = generate_with_tuned_lens_pop_restore(model, tuned_lens, layer_index, input_ids, gamma)
    pop_restore_time = now() - t0

print("manual draft time:", manual_time)
print("pop/restore draft time:", pop_restore_time)
print("manual faster by:", 1 - manual_time / pop_restore_time)

## 8. Full speculative loop using manual early forward

This mirrors your toy code, but uses the new draft function.

In [ ]:
gamma = 4
seqlen = 10
layer_index = 2
num_prefix_tokens = 10
num_of_examples = 100

n_accepted = 0
n_generated = 0

speculative_times = []
vanilla_times = []

draft_times = []
verify_times = []
sample_times = []

transformers.set_seed(42)

with torch.no_grad():
    for idx, example in enumerate(ds):
        if idx >= num_of_examples:
            break

        input_text = example["text"]
        input_ids = tokenizer(
            input_text,
            return_tensors="pt",
        )["input_ids"].to(device_type)[0][:num_prefix_tokens]

        all_ids = input_ids

        t_start_spec = now()

        while len(all_ids) - len(input_ids) < seqlen:
            remaining = seqlen - (len(all_ids) - len(input_ids))
            curr_gamma = min(gamma, remaining)

            # 1. Draft gamma tokens using manual early forward.
            t0 = now()
            small_model_output = generate_with_tuned_lens_manual(
                model=model,
                tuned_lens=tuned_lens,
                layer_index=layer_index,
                prefix_tokens=all_ids,
                gamma=curr_gamma,
            )
            draft_times.append(now() - t0)

            small_model_generated_tokens = small_model_output["sequences"][0, -curr_gamma:]

            q = torch.stack(small_model_output["scores"], dim=1)[0].float().softmax(dim=-1)

            q_of_generated = q[
                torch.arange(curr_gamma, device=device_type),
                small_model_generated_tokens,
            ]

            # 2. Verify with full model.
            t0 = now()
            full_seq = small_model_output["sequences"]
            attention_mask = torch.ones_like(full_seq, device=device_type)

            p_logits = model(
                full_seq,
                attention_mask=attention_mask,
                return_dict=True,
            ).logits

            # logits at positions before the generated draft tokens
            p = p_logits[:, -curr_gamma-1:-1, :].float().softmax(dim=-1)[0]

            p_of_generated = p[
                torch.arange(curr_gamma, device=device_type),
                small_model_generated_tokens,
            ]
            verify_times.append(now() - t0)

            # 3. Acceptance/rejection.
            t0 = now()
            ratio = torch.clamp(p_of_generated / torch.clamp(q_of_generated, min=1e-12), max=1.0)

            is_accepted = torch.rand(curr_gamma, device=device_type) < ratio

            index_to_reject = torch.argmin(
                torch.cat([
                    is_accepted,
                    torch.tensor([False], device=device_type),
                ]).to(torch.int)
            ).item()

            accepted_tokens = small_model_generated_tokens[:index_to_reject]

            if index_to_reject == curr_gamma:
                # all draft tokens accepted; sample the bonus token from p[-1]
                p_for_sample = p[-1]
            else:
                # rejection correction distribution
                p_for_sample = p[index_to_reject] - q[index_to_reject]
                p_for_sample = torch.clamp(p_for_sample, min=0)
                p_for_sample = p_for_sample / torch.clamp(p_for_sample.sum(), min=1e-12)

            n_accepted += index_to_reject
            n_generated += index_to_reject
            if index_to_reject < curr_gamma:
                n_generated += 1

            big_model_generated_token = torch.multinomial(p_for_sample, num_samples=1)

            # Avoid going past requested seqlen.
            new_tokens = torch.cat([accepted_tokens, big_model_generated_token])
            new_tokens = new_tokens[:remaining]

            all_ids = torch.cat([all_ids, new_tokens])
            sample_times.append(now() - t0)

        speculative_times.append(now() - t_start_spec)

        # 4. Vanilla generation baseline.
        t0 = now()

        attention_mask = torch.ones_like(input_ids[None], device=device_type)

        _ = model.generate(
            input_ids[None],
            attention_mask=attention_mask,
            max_new_tokens=len(all_ids) - len(input_ids),
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
        )

        vanilla_times.append(now() - t0)

        alpha = n_accepted / max(n_generated, 1)
        improvement = 1 - np.mean(speculative_times) / np.mean(vanilla_times)

        print(
            f"idx={idx:03d} "
            f"alpha={alpha:.2%} "
            f"improvement={improvement:.4f} "
            f"spec={np.mean(speculative_times):.4f}s "
            f"vanilla={np.mean(vanilla_times):.4f}s "
            f"draft={np.mean(draft_times):.4f}s "
            f"verify={np.mean(verify_times):.4f}s"
        )

## 9. Final summary

In [ ]:
summary = {
    "num_examples": len(speculative_times),
    "alpha": n_accepted / max(n_generated, 1),
    "mean_speculative_time": float(np.mean(speculative_times)) if speculative_times else None,
    "mean_vanilla_time": float(np.mean(vanilla_times)) if vanilla_times else None,
    "mean_improvement": float(1 - np.mean(speculative_times) / np.mean(vanilla_times)) if vanilla_times else None,
    "mean_draft_time": float(np.mean(draft_times)) if draft_times else None,
    "mean_verify_time": float(np.mean(verify_times)) if verify_times else None,
    "mean_sample_time": float(np.mean(sample_times)) if sample_times else None,
    "gamma": gamma,
    "layer_index": layer_index,
    "seqlen": seqlen,
    "num_prefix_tokens": num_prefix_tokens,
}

summary

## 10. What to try next

If manual early forward is faster than pop/restore but total speculative decoding is still slower, the next likely bottlenecks are:

1. `alpha` is too low
2. `gamma` is too small/large for that alpha
3. TunedLens full-vocab projection is expensive
4. verification still costs a full-model pass
5. T4 small-batch Python overhead dominates

Recommended quick grid:

```python
for layer_index in [2, 4, 6, 8, 10, 12]:
    for gamma in [2, 4, 6]:
        ...
```

Track:

```text
alpha
draft time
verify time
vanilla time
tokens/sec
```